In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from IPython.display import display, Javascript

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_districtflowname, get_data, get_data_outbkr, get_homebased_tag

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

In [ ]:
def pct_fmt(x):
    return 'nan' if pd.isna(x) else f'{x:,.2f}%'

def pct_compare_fmt(x):
    return 'nan' if pd.isna(x) else f'{x:,.1f}%'

In [ ]:
def preprocess(df):
    df = get_homebased_tag(df, tag_colname='hb_tag')
    df = get_districtflowname(df, taz_subarea=taz_subarea, taz_colname='otaz', new_colname='o_district')
    df = get_districtflowname(df, taz_subarea=taz_subarea, taz_colname='dtaz', new_colname='d_district')
    return df

# survey
data_fullsurvey['Trip'] = preprocess(data_fullsurvey['Trip'])

# bkrcast
data_daysim['Trip'] = preprocess(data_daysim['Trip'])

In [ ]:
summary_survey = data_fullsurvey['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

In [ ]:
def show_pivot_table(df):
    pivot_survey = (
    df
    .pivot_table(index='o_district', columns='d_district', values='trexpfac', aggfunc='sum', fill_value=0)
    .reindex(index=district_flow_name.values(), columns=district_flow_name.values())
    .fillna(0)
    )

    pivot_survey['Row Total'] = pivot_survey.sum(axis=1)
    pivot_survey.loc['Column Total'] = pivot_survey.sum()
    pivot_survey.columns.name = 'Destination'
    pivot_survey.index.name = 'Origin'

    display(pivot_survey.style.format('{:,.0f}').set_caption('Number of Trips'))
    # show as percent of grand total
    grand_total = pivot_survey.loc['Column Total', 'Row Total']
    if grand_total and grand_total != 0:
        pivot_pct = (pivot_survey / grand_total) * 100
        display(pivot_pct.style.format(pct_fmt).set_caption('Trip Share'))

# PSRC Region

## All

In [ ]:
show_pivot_table(summary_survey)

In [ ]:
show_pivot_table(summary_daysim)

## HBW

In [ ]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

In [ ]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

## HBO

In [ ]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

In [ ]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

## NHB

In [ ]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

In [ ]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

# In-BKR Households

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)

In [ ]:
summary_survey = data_fullsurvey_bkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim_bkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

## All

In [ ]:
show_pivot_table(summary_survey)

In [ ]:
show_pivot_table(summary_daysim)

## HBW

In [ ]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

In [ ]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

## HBO

In [ ]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

In [ ]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

## NHB

In [ ]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

In [ ]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

# Outside-BKR Households

In [ ]:
data_daysim_outbkr, data_survey_outbkr, data_fullsurvey_outbkr = \
    get_data_outbkr(data1=data_daysim, data2=data_survey, data3=data_fullsurvey, taz_subarea=taz_subarea)

In [ ]:
summary_survey = data_fullsurvey_outbkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim_outbkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

## All

In [ ]:
show_pivot_table(summary_survey)

In [ ]:
show_pivot_table(summary_daysim)

## HBW

In [ ]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

In [ ]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

## HBO

In [ ]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

In [ ]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

## NHB

In [ ]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

In [ ]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)